In [1]:
import sys
from pathlib import Path

# find the project root (the folder that contains src/features.py) and put src/ on the import path
HERE = Path.cwd().resolve()
ROOT = next(p for p in [HERE, *HERE.parents] if (p / "src" / "features.py").exists())
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [2]:
import numpy as np
import pandas as pd

import contract  
import data 


In [3]:
df = data.read_raw()

In [4]:
df.head(2)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01


In [5]:
df["arrival_date"] = data.arrival_date(df)
df["nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]

In [6]:
df.head(2)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date,nights
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01,2015-07-01,0
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01,2015-07-01,0


In [7]:
df.shape

(119390, 34)

In [8]:
df['arrival_date'].min()

Timestamp('2015-07-01 00:00:00')

In [9]:
df['arrival_date'].max()

Timestamp('2017-08-31 00:00:00')

In [10]:
round(df['is_canceled'].sum()/df['is_canceled'].count()*100,2)

np.float64(37.04)

In [11]:
hotel_df = df.groupby('hotel').agg(
    total_bookings = ('hotel','count'),
    total_cancelations = ('is_canceled','sum')
).reset_index()

In [12]:
hotel_df

,hotel,total_bookings,total_cancelations
0,City Hotel,79330,33102
1,Resort Hotel,40060,11122


In [13]:
hotel_df['percentage'] = round(hotel_df['total_cancelations']/hotel_df['total_bookings']*100,2)
hotel_df

,hotel,total_bookings,total_cancelations,percentage
0,City Hotel,79330,33102,41.73
1,Resort Hotel,40060,11122,27.76


The Claim is False, As in the given data it is said that 25% is the cancelation rate

In [14]:
deposit_df = df.groupby("deposit_type").agg(
        total_bookings = ('hotel','count'),
        total_cancelations = ('is_canceled','sum')
).reset_index()
deposit_df['percentage'] =  round(deposit_df['total_cancelations']/deposit_df['total_bookings']*100,2)
deposit_df

,deposit_type,total_bookings,total_cancelations,percentage
0,No Deposit,104641,29694,28.38
1,Non Refund,14587,14494,99.36
2,Refundable,162,36,22.22


The Claim is True, As in the given data it is said that the 99% is the cancelation rate for Non Refund

In [15]:
segment_df = df.groupby("market_segment").agg(
        total_bookings = ('hotel','count'),
        total_cancelations = ('is_canceled','sum')
).reset_index()
segment_df['percentage'] =  round(segment_df['total_cancelations']/segment_df['total_bookings']*100,2)
segment_df

,market_segment,total_bookings,total_cancelations,percentage
0,Aviation,237,52,21.94
1,Complementary,743,97,13.06
2,Corporate,5295,992,18.73
3,Direct,12606,1934,15.34
4,Groups,19811,12097,61.06
5,Offline TA/TO,24219,8311,34.32
6,Online TA,56477,20739,36.72
7,Undefined,2,2,100.00


The Claim is False, As they said that cancelation across the all merketing segments are same, but we found that Groups cancelations is 61% and direct is 15%

In [16]:
train, choose, proof = contract.split_periods(df)

In [17]:
train.shape

(49157, 34)

In [18]:
int(train['is_canceled'].sum())

17585

In [19]:
train_df = train.groupby("hotel").agg(
        total_bookings = ('hotel','count'),
        total_cancelations = ('is_canceled','sum')
).reset_index()
train_df['percentage'] =  round(train_df['total_cancelations']/train_df['total_bookings']*100,2)
train_df

,hotel,total_bookings,total_cancelations,percentage
0,City Hotel,31623,13175,41.66
1,Resort Hotel,17534,4410,25.15


In [20]:
choose.shape

(29546, 34)

In [21]:
int(choose['is_canceled'].sum())

10894

In [22]:
choose_df = choose.groupby("hotel").agg(
        total_bookings = ('hotel','count'),
        total_cancelations = ('is_canceled','sum')
).reset_index()
choose_df['percentage'] =  round(choose_df['total_cancelations']/choose_df['total_bookings']*100,2)
choose_df

,hotel,total_bookings,total_cancelations,percentage
0,City Hotel,20199,8236,40.77
1,Resort Hotel,9347,2658,28.44


In [23]:
proof.shape

(40687, 34)

In [24]:
int(proof['is_canceled'].sum())

15745

In [25]:
proof_df = proof.groupby("hotel").agg(
        total_bookings = ('hotel','count'),
        total_cancelations = ('is_canceled','sum')
).reset_index()
proof_df['percentage'] =  round(proof_df['total_cancelations']/proof_df['total_bookings']*100,2)
proof_df

,hotel,total_bookings,total_cancelations,percentage
0,City Hotel,27508,11691,42.50
1,Resort Hotel,13179,4054,30.76


In [26]:
#ADR value is negitive
int((df["adr"] < 0).sum())


1

In [27]:
#ADR value is equal to Zero
int((df['adr'].eq(0).sum()))

1959

In [28]:
# Nights equal to Zero
int((df['nights'].eq(0).sum()))

715

In [29]:
# Adults equal to Zero (Not possible)
int(df['adults'].eq(0).sum())

403

In [30]:
# Children missing
int(df['children'].isna().sum())

4

In [31]:
# we are converting the value of adr which is negitive to 0
df['adr'] = np.where(df['adr'].lt(0), 0 , df['adr'])
int((df["adr"] < 0).sum())

0